In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

# 1. Generate Dataset with high correlation
# n_features=10, effective_rank=2 ensures high multicollinearity
X, y = make_regression(n_samples=1000, n_features=10, effective_rank=2, noise=10, random_state=42)

# Scale data (Critical for Gradient Descent convergence)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 2. Ridge Regression Class using Gradient Descent
class RidgeRegressionGD:
    def __init__(self, learning_rate=0.01, lambda_param=1.0, epochs=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param
        self.epochs = epochs
        self.weights = None
        self.bias = None
        self.cost_history = []

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        for _ in range(self.epochs):
            # Hypothesis
            y_predicted = np.dot(X, self.weights) + self.bias
            
            # Gradients
            dw = (1 / n_samples) * (np.dot(X.T, (y_predicted - y)) + 2 * self.lambda_param * self.weights)
            db = (1 / n_samples) * np.sum(y_predicted - y) # Bias is usually not regularized
            
            # Update
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
            
            # Calculate Cost
            cost = (1 / (2 * n_samples)) * np.sum((y_predicted - y) ** 2) + \
                   (self.lambda_param * np.sum(self.weights ** 2))
            self.cost_history.append(cost)

    def predict(self, X):
        return np.dot(X, self.weights) + self.bias

# 3. Hyperparameter Tuning
learning_rates = [0.0001, 0.001, 0.01, 0.1, 1] # Note: 10 often causes divergence depending on scaling
regularizations = [1e-15, 1e-10, 1e-5, 1e-3, 0, 1, 10, 20]

best_r2 = -float('inf')
best_params = {}
best_model = None

print(f"{'LR':<10} {'Lambda':<10} {'Cost':<15} {'R2 Score':<10}")
print("-" * 50)

for lr in learning_rates:
    for lam in regularizations:
        model = RidgeRegressionGD(learning_rate=lr, lambda_param=lam, epochs=500)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        
        r2 = r2_score(y_test, preds)
        final_cost = model.cost_history[-1]
        
        if r2 > best_r2:
            best_r2 = r2
            best_params = {'lr': lr, 'lambda': lam, 'cost': final_cost}
            best_model = model
            
        # Print a few examples or all to track
        # print(f"{lr:<10} {lam:<10} {final_cost:.4f}          {r2:.4f}")

print("\nBest Parameters Found:")
print(f"Learning Rate: {best_params['lr']}")
print(f"Regularization (Lambda): {best_params['lambda']}")
print(f"Minimum Cost: {best_params['cost']:.4f}")
print(f"Maximum R2 Score: {best_r2:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

# (a) Load and Pre-process
url = "https://raw.githubusercontent.com/kassambara/datarium/master/data/Hitters.csv"
try:
    df = pd.read_csv(url)
except:
    print("Could not load URL. Ensure 'Hitters.csv' is in your local directory.")
    # df = pd.read_csv('Hitters.csv') # Uncomment if running locally

# Drop Null Values (Salary usually has NaNs in this dataset)
df = df.dropna()

# Categorical to Numerical (One-Hot Encoding or simple mapping)
# The Hitters dataset has columns 'League', 'Division', 'NewLeague'
df = pd.get_dummies(df, columns=['League', 'Division', 'NewLeague'], drop_first=True)

# (b) Separate Input/Output and Scale
X = df.drop('Salary', axis=1)
y = df['Salary']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# (c) Fit Models
# 1. Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

# 2. Ridge Regression (alpha = 0.5748)
ridge_reg = Ridge(alpha=0.5748)
ridge_reg.fit(X_train, y_train)

# 3. Lasso Regression (alpha = 0.5748)
lasso_reg = Lasso(alpha=0.5748)
lasso_reg.fit(X_train, y_train)

# (d) Evaluate
models = {'Linear': lin_reg, 'Ridge': ridge_reg, 'Lasso': lasso_reg}

print(f"{'Model':<10} {'RMSE':<15} {'R2 Score':<10}")
print("-" * 40)

for name, model in models.items():
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    r2 = r2_score(y_test, pred)
    print(f"{name:<10} {rmse:.4f}          {r2:.4f}")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import RidgeCV, LassoCV
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load Boston Dataset (Alternative method due to sklearn deprecation)
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

X_train, X_test, y_train, y_test = train_test_split(data, target, test_size=0.2, random_state=42)

# Scaling is important for Regularization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. RidgeCV
# Scans alphas automatically (e.g., 0.1, 1.0, 10.0)
ridge_cv = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 20.0, 50.0], cv=5)
ridge_cv.fit(X_train_scaled, y_train)

print(f"RidgeCV Best Alpha: {ridge_cv.alpha_}")
print(f"RidgeCV R2 Score: {ridge_cv.score(X_test_scaled, y_test):.4f}")

# 2. LassoCV
lasso_cv = LassoCV(cv=5, random_state=42)
lasso_cv.fit(X_train_scaled, y_train)

print(f"LassoCV Best Alpha: {lasso_cv.alpha_}")
print(f"LassoCV R2 Score: {lasso_cv.score(X_test_scaled, y_test):.4f}")

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

# Load Data
iris = load_iris()
X = iris.data
y = iris.target # Classes: 0, 1, 2

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Dictionary to store our trained binary models
models = {}
classes = np.unique(y) # [0, 1, 2]

print("Training One-vs-Rest Models...")

# Step 1: Train one binary classifier per class
for c in classes:
    # Create binary target: 1 if class is c, else 0
    y_train_binary = np.where(y_train == c, 1, 0)
    
    # Train Logistic Regression
    model = LogisticRegression(solver='lbfgs')
    model.fit(X_train, y_train_binary)
    
    models[c] = model
    print(f"Model for Class {c} trained.")

# Step 2: Prediction Strategy
final_predictions = []

for sample in X_test:
    # Get probability of being "True" from all 3 models
    # model.predict_proba returns [prob_0, prob_1]. We want prob_1.
    probs = {}
    for c in classes:
        probs[c] = models[c].predict_proba([sample])[0][1]
    
    # Argmax: Select class with highest probability
    best_class = max(probs, key=probs.get)
    final_predictions.append(best_class)

# Evaluation
print("\nAccuracy:", accuracy_score(y_test, final_predictions))
print("\nClassification Report:\n", classification_report(y_test, final_predictions))